[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter09_autograd_calculus_for_free/notebook.ipynb)

# Chapter 9: Autograd: Calculus for Free

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

In Chapter 7 you worked out the gradients of a neural network by hand. It took a page of careful reasoning, and the network had nine weights in it. Now imagine doing that again after changing one activation. Then again after adding a layer. Then again after changing the loss. For most of the history of neural networks that is exactly what researchers did, and a new architecture meant a fresh page of derivatives, checked slowly against nudged measurements to catch the inevitable sign error.

That tedium shaped the field in a way that is easy to miss. If trying a new idea costs a week of error prone calculus, you try fewer ideas, and the ones you try stay close to what already works. The frameworks that arrived in the 2010s removed that cost, and PyTorch took a particular approach to it. Rather than asking you to describe your network in advance so the system could analyse it, PyTorch simply watches your ordinary Python code as it runs, writes down every operation it performs, and then replays that record backwards.

The result is a feature called **autograd**, short for automatic gradients, and it is the real reason people use a framework. This chapter is where you check that it works, by asking it for the exact numbers you derived yourself in Chapter 7. Before you read on, try to guess the answers to these four questions.

1. **For a tensor to know its own gradient, what would it have to remember?**
2. **Who decides when the backward pass actually happens?**
3. **After you have used a gradient to update a weight, what should happen to it?**
4. **When would you want to switch this machinery off?**

## The Array That Remembers

An ordinary tensor is just numbers. A tensor with `requires_grad=True` is a tensor that keeps a record of everything done to it, so that it can later be asked how it affected a result. Start with the simplest function there is, squaring a number.

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
print(y)
print(x.grad)

Two things to notice. The result prints with `grad_fn` attached, which is PyTorch saying this number came from a power operation and it remembers that. And `x.grad` is `None`, because nothing has asked for a gradient yet. The asking is a single call.

In [ ]:
y.backward()
print(x.grad)

The slope of $x^2$ is $2x$, which at $x=3$ is 6, and that is what PyTorch found without being told the formula. It is worth sitting with how little you had to do. You wrote the forward calculation in plain Python, and the derivative came back for free.

**Mark a tensor with `requires_grad=True` and PyTorch records every operation applied to it, so that calling `backward()` on a result fills in that tensor's `grad`.**

The word `backward` is literal. The recording is replayed from the result back to the inputs, applying the chain rule from Chapter 7 at each step, which is the same journey you made by hand with the error signals.

## How Autograd Builds a Graph

The record PyTorch keeps is called the **computational graph**. It is built as your code runs, one operation at a time, and it is thrown away as soon as it has been used. Think of it as a tape in a recorder rather than a blueprint drawn in advance.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = a * 3
c = b ** 2
print(type(b.grad_fn).__name__, type(c.grad_fn).__name__)

Each intermediate value knows which operation produced it. The tensor `b` remembers it came from a multiplication, and `c` remembers it came from a power, so the chain from `a` to `c` is fully recorded. Figure 9.1 draws it, forwards and backwards.

> 🖼️ **Figure 9.1:** The tape for $c=(3a)^2$. The forward pass records each operation, and `backward()` replays the record in reverse, multiplying the links exactly as the chain rule in Chapter 7 did.

Working the chain by hand gives $c = (3a)^2$, so the slope is $2 \times (3a) \times 3$, which at $a=2$ is $2 \times 6 \times 3 = 36$. The next cell checks it.

In [ ]:
c.backward()
print(a.grad)

Because the tape is built while your code runs, anything Python can do, the network can do. A loop whose length changes, an `if` that takes a different branch on different inputs, a `print` in the middle of a layer, all of it works, because the graph is simply whatever actually happened this time. That is the design choice PyTorch inherited from Torch, and it is why debugging a PyTorch model feels like debugging ordinary code.

## Reproducing Your Hand Derived Gradients

Now for the real test. In Chapter 7 you took a network with two inputs, two hidden neurons and one output, fed it the input 1 and 0 with a target of 1, and derived every gradient by hand. This section rebuilds that exact network in PyTorch and asks autograd for the same numbers.

The weights are the same ones, now marked so that PyTorch tracks them.

In [ ]:
W1 = torch.tensor([[0.5, -0.5],
                   [0.5,  0.5]], requires_grad=True)
b1 = torch.tensor([0.0, 0.0], requires_grad=True)
W2 = torch.tensor([1.0, -1.0], requires_grad=True)
b2 = torch.tensor(0.0, requires_grad=True)

x = torch.tensor([1.0, 0.0])
target = torch.tensor(1.0)

The forward pass is the same three lines as Chapter 7, with `torch.sigmoid` in place of the one you wrote yourself.

In [ ]:
h = torch.sigmoid(W1 @ x + b1)
y = torch.sigmoid(W2 @ h + b2)
loss = (y - target) ** 2
print(h.detach().numpy().round(4), round(y.item(), 4), round(loss.item(), 4))

Identical to Chapter 7, where the two hidden neurons both gave 0.6225, the output weights cancelled to give exactly 0.5, and the loss was 0.25. Now the part that took a page by hand.

In [ ]:
loss.backward()
print(W2.grad.numpy().round(4), round(b2.grad.item(), 4))

Those are your numbers. In Chapter 7 you worked out the output error signal as $-0.25$, multiplied it by each hidden output of 0.6225 to get $-0.1556$ for the weights, and noted that the bias inherits the error signal unchanged. PyTorch agrees exactly. The first layer is the better test, because that is where the error had to travel backwards through the hidden neurons.

In [ ]:
print(W1.grad.numpy().round(4))
print(b1.grad.numpy().round(4))

Every number matches, including the detail that made the point in Chapter 7, which is that the whole second column is zero because the second input was 0 and those weights had no influence. PyTorch did not have to be told that. It fell out of the same chain rule you applied yourself. Table 9.1 puts the two side by side.

**Table 9.1: A page of hand derivation against one call to `backward()`. This is the moment to start trusting the framework, because you checked it rather than being told to.**

| Gradient | Chapter 7, by hand | Chapter 9, autograd |
|---|---|---|
| Output weights | $-0.1556$, $-0.1556$ | $-0.1556$, $-0.1556$ |
| Output bias | $-0.25$ | $-0.25$ |
| First layer, first column | $-0.0588$, $0.0588$ | $-0.0588$, $0.0588$ |
| First layer, second column | 0, 0 | 0, 0 |
| Hidden biases | $-0.0588$, $0.0588$ | $-0.0588$, $0.0588$ |

The same holds for Chapter 6. There you measured the café's slope by nudging, got $-835.67$, and matched it with a formula you differentiated yourself. Autograd gives a third independent answer.

In [ ]:
temp = torch.tensor([20., 24., 28., 30., 33., 35.])
drinks = torch.tensor([41., 47., 58., 59., 67., 69.])

w = torch.tensor(1.5, requires_grad=True)
mse = ((w * temp - drinks) ** 2).mean()
mse.backward()
print(round(w.grad.item(), 2))

Nudging, calculus and autograd all agree. **Autograd is not a different method from the backpropagation you learned, it is the same chain rule applied by a machine that never makes a sign error.**

## When to Turn Autograd Off

Autograd is on whenever a tracked tensor is involved, and there are three situations where you must take charge of it. Getting these wrong produces bugs that do not crash, which are the worst kind.

The first is the one that catches everybody. Gradients do not replace each other, they add up.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    loss = w ** 2
    loss.backward()
    print(i, w.grad.item())

The true gradient is 2 every time, but it piles up to 4 and then 6. A network trained like this would take steps that grow larger every round for no reason, and nothing would announce the problem. The fix is to clear the gradient before each backward pass.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    if w.grad is not None:
        w.grad.zero_()
    loss = w ** 2
    loss.backward()
    print(i, w.grad.item())

This accumulation is deliberate rather than a flaw, because it lets you add up gradients from several batches before stepping. Chapter 10 introduces an optimizer with a `zero_grad()` method that does the clearing for you, and from then on you will simply remember to call it.

The second situation is updating the weights. The update itself is arithmetic on tracked tensors, so autograd would happily record it as part of the next graph, which is wrong and wasteful. Wrapping it in `torch.no_grad()` tells PyTorch to stop recording.

In [ ]:
p = torch.tensor(2.0, requires_grad=True)
tracked = p * 5
with torch.no_grad():
    untracked = p * 5
print(tracked.requires_grad, untracked.requires_grad)

The third is when you simply want a number out, for printing, plotting or storing, which is what `.detach()` is for. It hands back the same values with the recording stripped off, and it is why the forward pass earlier in this chapter printed `h.detach().numpy()` rather than `h.numpy()`.

In [ ]:
print(tracked.detach().requires_grad)

Put all three together and you can write Chapter 6's gradient descent with no hand derived slope anywhere in it. The next cell rolls down the café bowl exactly as before, except that autograd supplies the slope.

In [ ]:
w = torch.tensor(1.5, requires_grad=True)
for step in range(31):
    loss = ((w * temp - drinks) ** 2).mean()
    if step in (0, 1, 2, 5, 10, 30):
        print(step, round(w.item(), 4), round(loss.item(), 3))
    loss.backward()
    with torch.no_grad():
        w -= 0.0003 * w.grad
    w.grad.zero_()

Those are the same six lines Chapter 6 printed, digit for digit, from code that contains no calculus at all. Table 9.2 gathers the three controls and the situations that call for them.

**Table 9.2: The three controls, the four situations that need them, and the cost of forgetting each one. Note that `no_grad` earns its place twice.**

| Situation | What to use | What goes wrong without it |
|---|---|---|
| Before each backward pass | `grad.zero_()` | Gradients pile up and steps grow |
| Updating the weights | `with torch.no_grad():` | The update is recorded as part of the next graph |
| Printing or plotting a value | `.detach()` | An error, or a graph kept alive in memory |
| Measuring on validation data | `with torch.no_grad():` | Wasted memory and time on a graph never used |

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Check a cube.** Build `x = torch.tensor(2.0, requires_grad=True)`, compute `y = x ** 3`, call `backward()` and check that `x.grad` equals the 12 that $3x^2$ predicts.
2. **Feed the other input.** Change `x` in the Chapter 7 network to `[0.0, 1.0]`, run the forward and backward passes again, and confirm that this time the first column of `W1.grad` is zero instead of the second.
3. **Forget to zero.** Delete the `w.grad.zero_()` line from the last training loop and run it again. Watch where the weight ends up, and explain the result using the accumulation you saw earlier.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What does `requires_grad=True` actually change about a tensor?**
2. **What is the computational graph, and when is it built?**
3. **Why did `x.grad` start as `None`?**
4. **What happens if you call `backward()` three times without clearing gradients?**
5. **Why must a weight update be wrapped in `torch.no_grad()`?**

### Answers

1. It makes the tensor record every operation applied to it, so a later `backward()` call can work out its gradient.
2. It is the record of operations that produced a result. It is built while your code runs, and discarded once it has been used.
3. Because no backward pass had happened yet, so there was no gradient to hold.
4. The gradients add up instead of replacing each other, so you get 2, then 4, then 6 for a true gradient of 2.
5. Because the update is arithmetic on tracked tensors, so without it autograd would record the update itself as part of the next graph.

## What Comes Next

You can now let PyTorch find every gradient in a network, you have checked its answers against your own by hand, and you know where you have to take charge of it.

- **Chapter 10, Building a Network with nn.Module,** replaces loose weight matrices with a proper model object that keeps track of its own parameters.
- **Chapter 11, The Training Loop,** puts the forward pass, the loss, the backward pass and the update into the five step loop you will use for the rest of this book.

Chapter 10 begins with a practical annoyance. The network in this chapter had four separate tensors to pass around, and a real one has hundreds.